# Import required library

In [2]:
import epi_utils as eu
import pandas as pd
import pyranges as pr
import glob
from tqdm.auto import tqdm

In [3]:
chrom_list = pd.read_csv("dataset/chrom_list.csv")["Chromosome"].to_list()

# Splitting Histone Data

In [4]:
# Load histone file
histone_df = eu.load_histone_files()
display(histone_df.head())

,chrom,chromStart,chromEnd,name,length,type
0,chr10,119808,119954,chr10_173,146,h3k4me3
1,chr10,119956,120102,chr10_174,146,h3k4me3
2,chr10,122100,122246,chr10_185,146,h3k4me3
3,chr10,122308,122454,chr10_186,146,h3k4me3
4,chr10,180346,180492,chr10_489,146,h3k4me3


In [5]:
for chrom in tqdm(chrom_list):
    df = histone_df[histone_df['chrom'] == chrom]
    print(f"Processing {chrom} with {len(df.index)} data")
    df.to_csv("dataset/histone/" + chrom + ".csv", index=False)

  0%|          | 0/24 [00:00<?, ?it/s]

Processing chr1 with 30373 data
Processing chr2 with 24677 data
Processing chr3 with 19425 data
Processing chr4 with 14442 data
Processing chr5 with 16617 data
Processing chr6 with 18508 data
Processing chr7 with 16546 data
Processing chr8 with 13445 data
Processing chr9 with 13472 data
Processing chr10 with 14519 data
Processing chr11 with 15712 data
Processing chr12 with 15929 data
Processing chr13 with 7811 data
Processing chr14 with 10687 data
Processing chr15 with 10076 data
Processing chr16 with 13046 data
Processing chr17 with 16532 data
Processing chr18 with 6333 data
Processing chr19 with 15234 data
Processing chr20 with 8065 data
Processing chr21 with 4116 data
Processing chr22 with 6634 data
Processing chrX with 6424 data
Processing chrY with 463 data


# Splitting NCBI RefSeq Data

## Load the NCBI Ref Seq Data

In [6]:
hg19ncbiRefSeq_df = eu.load_ncbiRefSeq()
display(hg19ncbiRefSeq_df.head())

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,transcript_id,gene_name,exon_number,exon_id,tss
0,chrMT,ncbiRefSeq.2021-05-17,transcript,15955,16023,.,-,.,TRNP,rna-TRNP,TRNP,NaN,NaN,16023
1,chrMT,ncbiRefSeq.2021-05-17,exon,15955,16023,.,-,.,TRNP,rna-TRNP,TRNP,1,rna-TRNP.1,16023
2,chrMT,ncbiRefSeq.2021-05-17,transcript,15887,15953,.,+,.,TRNT,rna-TRNT,TRNT,NaN,NaN,15887
3,chrMT,ncbiRefSeq.2021-05-17,exon,15887,15953,.,+,.,TRNT,rna-TRNT,TRNT,1,rna-TRNT.1,15887
4,chrMT,ncbiRefSeq.2021-05-17,transcript,14746,15887,.,+,.,CYTB,YP_003024038.1,CYTB,NaN,NaN,14746


In [7]:
display(hg19ncbiRefSeq_df.shape)

(2093542, 14)

## Splitting into each `Chromosome`

In [9]:
print(f"Splitting into {len(chrom_list)} files.")

num_file = 1

for chrom in tqdm(chrom_list):
    df = hg19ncbiRefSeq_df[hg19ncbiRefSeq_df['Chromosome'] == chrom]
    print(f"File {num_file}: Processing {chrom} with {len(df.index)} data")
    df.to_csv(f"dataset/ncbiRefSeq/raw/{chrom}.csv", index=False)
    num_file+=1

Splitting into 24 files.


  0%|          | 0/24 [00:00<?, ?it/s]

File 1: Processing chr1 with 179608 data
File 2: Processing chr2 with 149205 data
File 3: Processing chr3 with 130850 data
File 4: Processing chr4 with 79002 data
File 5: Processing chr5 with 82697 data
File 6: Processing chr6 with 97005 data
File 7: Processing chr7 with 96481 data
File 8: Processing chr8 with 76169 data
File 9: Processing chr9 with 80498 data
File 10: Processing chr10 with 93765 data
File 11: Processing chr11 with 116373 data
File 12: Processing chr12 with 105694 data
File 13: Processing chr13 with 35299 data
File 14: Processing chr14 with 56060 data
File 15: Processing chr15 with 62803 data
File 16: Processing chr16 with 73227 data
File 17: Processing chr17 with 94307 data
File 18: Processing chr18 with 34398 data
File 19: Processing chr19 with 95761 data
File 20: Processing chr20 with 42015 data
File 21: Processing chr21 with 20311 data
File 22: Processing chr22 with 39819 data
File 23: Processing chrX with 59333 data
File 24: Processing chrY with 8468 data


## Combining similar `Chromosome`

In [10]:
# Flatten the glob result
def flatten_concatenation(matrix):
    flat_list = []
    for row in matrix:
        flat_list += row
    return flat_list

In [11]:
for chrom in tqdm(chrom_list):
    print(f"Processing chromosome: {chrom}")
    file_list = []
    file_list.append(glob.glob(f"dataset/ncbiRefSeq/raw/{chrom}.csv"))
    file_list.append(glob.glob(f"dataset/ncbiRefSeq/raw/{chrom}_*.csv"))
    flat_file_list = flatten_concatenation(file_list)

    data = []
    for filename in flat_file_list:
        print(f"Processing file: {filename}")
        df = pd.read_csv(filename)
        data.append(df)

    df_out = pd.concat(data, axis=0, ignore_index=False)
    df_out.to_csv(f"dataset/ncbiRefSeq/merged/{chrom}.csv", index=False)

print("Splitting ncbiRefSeq FINISHED!")

  0%|          | 0/24 [00:00<?, ?it/s]

Processing chromosome: chr1
Processing file: dataset/ncbiRefSeq/raw/chr1.csv
Processing chromosome: chr2
Processing file: dataset/ncbiRefSeq/raw/chr2.csv
Processing chromosome: chr3
Processing file: dataset/ncbiRefSeq/raw/chr3.csv
Processing chromosome: chr4
Processing file: dataset/ncbiRefSeq/raw/chr4.csv
Processing chromosome: chr5
Processing file: dataset/ncbiRefSeq/raw/chr5.csv
Processing chromosome: chr6
Processing file: dataset/ncbiRefSeq/raw/chr6.csv
Processing chromosome: chr7
Processing file: dataset/ncbiRefSeq/raw/chr7.csv
Processing chromosome: chr8
Processing file: dataset/ncbiRefSeq/raw/chr8.csv
Processing chromosome: chr9
Processing file: dataset/ncbiRefSeq/raw/chr9.csv
Processing chromosome: chr10
Processing file: dataset/ncbiRefSeq/raw/chr10.csv
Processing chromosome: chr11
Processing file: dataset/ncbiRefSeq/raw/chr11.csv
Processing chromosome: chr12
Processing file: dataset/ncbiRefSeq/raw/chr12.csv
Processing chromosome: chr13
Processing file: dataset/ncbiRefSeq/raw/c

# Counting Histone in TSS

In [ ]:
# Counting chrY
chrom = 'chrY'

histone_chr_df = pd.read_csv(f"dataset/histone/{chrom}.csv")
ncbiRefSeq_chr_df = pd.read_csv(f"dataset/ncbiRefSeq/merged/{chrom}.csv")

display(histone_chr_df.head())
print(histone_chr_df.shape)
display(ncbiRefSeq_chr_df.head())
print(ncbiRefSeq_chr_df.shape)

In [ ]:
tqdm.pandas()

In [ ]:
# Counting Histone
ncbiRefSeq_chr_df.loc[:, "histone_count"] = ncbiRefSeq_chr_df.progress_apply(\
                            lambda row: eu.count_histone(histone_chr_df, row), axis=1)

In [ ]:
display(ncbiRefSeq_chr_df.head())

In [ ]:
ncbiRefSeq_chr_df.to_csv(f"dataset/histone_count/{chrom}.csv", index=False)

In [ ]:
ax = ncbiRefSeq_chr_df.hist(column=["histone_count"])

In [ ]:
histone_path = "dataset/histone/"
ncbiRefSeq_path = "dataset/ncbiRefSeq/merged/"
output_path = "dataset/histone_count/"

In [ ]:
# chrUn and chrMT histone count should be 0

chrom = 'chrMT'

# Reading file
ncbiRefSeq_chr_df = pd.read_csv(f"{ncbiRefSeq_path}{chrom}.csv")

# Histone count = 0
ncbiRefSeq_chr_df.loc[:, "histone_count"] = 0
display(ncbiRefSeq_chr_df.head())

# Saving the result
ncbiRefSeq_chr_df.to_csv(f"{output_path}{chrom}.csv", index=False)

In [ ]:
# Run the counting_histone.py to get the result
# Getting all the results back to dataframe

file_list = []
file_list.append(glob.glob(f"{output_path}*.csv"))
flat_file_list = flatten_concatenation(file_list)
print(flat_file_list)

In [ ]:
data = []
for filename in tqdm(flat_file_list):
    df = pd.read_csv(filename)
    data.append(df)

df_out = pd.concat(data, axis=0, ignore_index=False)
df_out.to_csv(f"{output_path}histone_count.csv", index=False)

In [ ]:
display(df_out.head())
display(df_out.shape)

In [ ]:
ax = df_out.hist(column=["histone_count"])

In [ ]:
# Check the frequency of each histone count
df_out.groupby(['histone_count']).size().reset_index(name='counts')

# Splitting HepG2 data

## Loading and Preparing the Data

In [12]:
# Loading HepG2 data
hepg2_exp = pd.read_csv("dataset/GSM3718064_HepG2_exp.txt", sep="\t")

In [13]:
# Splitting chromosome name and the position
hepg2_exp[["chrom", "chromPos"]] = hepg2_exp["locus"].str.split(':', expand=True)
display(hepg2_exp.head())

,test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chrom,chromPos
0,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,69090-70008
1,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,0.047392,0.034159,-0.472389,0.000000,1.00000,1.000000,no,chr1,323891-328581
2,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,367658-368597
3,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,2.468570,2.805800,0.184736,0.455074,0.63585,0.999565,no,chr1,761585-794889
4,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,840263-843900


In [14]:
# Splitting start and end position
hepg2_exp[["chromStart", "chromEnd"]] = hepg2_exp["chromPos"].str.split('-', expand=True)
display(hepg2_exp.head())

,test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chrom,chromPos,chromStart,chromEnd
0,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,69090-70008,69090,70008
1,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,0.047392,0.034159,-0.472389,0.000000,1.00000,1.000000,no,chr1,323891-328581,323891,328581
2,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,367658-368597,367658,368597
3,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,2.468570,2.805800,0.184736,0.455074,0.63585,0.999565,no,chr1,761585-794889,761585,794889
4,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,840263-843900,840263,843900


In [15]:
# Correcting the Data Types
hepg2_exp = hepg2_exp.astype({'chromStart': 'int32', 'chromEnd': 'int32'})

In [16]:
# Find the gene's length
hepg2_exp["gene_length"] = hepg2_exp["chromEnd"] - hepg2_exp["chromStart"]
display(hepg2_exp)

,test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chrom,chromPos,chromStart,chromEnd,gene_length
0,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,69090-70008,69090,70008,918
1,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,0.047392,0.034159,-0.472389,0.000000,1.00000,1.000000,no,chr1,323891-328581,323891,328581,4690
2,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,367658-368597,367658,368597,939
3,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,2.468570,2.805800,0.184736,0.455074,0.63585,0.999565,no,chr1,761585-794889,761585,794889,33304
4,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,840263-843900,840263,843900,3637
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30027,XLOC_030028,XLOC_030028,-,chrY:13319431-13324829,hepg2_hr2,hepg2_hr3,OK,0.337879,0.457897,0.438516,1.137930,0.25685,0.999565,no,chrY,13319431-13324829,13319431,13324829,5398
30028,XLOC_030029,XLOC_030029,-,chrY:13325034-13326120,hepg2_hr2,hepg2_hr3,NOTEST,0.396888,0.357658,-0.150148,0.000000,1.00000,1.000000,no,chrY,13325034-13326120,13325034,13326120,1086
30029,XLOC_030030,XLOC_030030,-,chrY:13331089-13332358,hepg2_hr2,hepg2_hr3,OK,0.477227,0.127331,-1.906090,-1.861920,0.12490,0.999565,no,chrY,13331089-13332358,13331089,13332358,1269
30030,XLOC_030031,XLOC_030031,-,chrY:15053270-15053343,hepg2_hr2,hepg2_hr3,OK,123998.000000,71768.700000,-0.788894,-2.116040,0.04075,0.999565,no,chrY,15053270-15053343,15053270,15053343,73


In [ ]:
# # Find the gene length distribution
# ax = hepg2_exp.hist(column=["gene_length"])

In [ ]:
# # Find the frequency of gene length
# hepg2_exp.groupby(['gene_length']).size().reset_index(name='counts')

In [ ]:
# hepg2_exp.groupby(['gene_length'])['gene_length'].size().reset_index(name='counts').to_csv("dataset/HepG2/gene_length.csv", index=False)

## Splitting into each Chromosome

In [17]:
print(f"Splitting into {len(chrom_list)} files.")

num_file = 1

for chrom in tqdm(chrom_list):
    df = hepg2_exp[hepg2_exp['chrom'] == chrom]
    print(f"File {num_file}: Processing {chrom} with {len(df.index)} data")
    df.to_csv(f"dataset/HepG2/raw/{chrom}.csv", index=False)
    num_file+=1

Splitting into 24 files.


  0%|          | 0/24 [00:00<?, ?it/s]

File 1: Processing chr1 with 2867 data
File 2: Processing chr2 with 1880 data
File 3: Processing chr3 with 1581 data
File 4: Processing chr4 with 1137 data
File 5: Processing chr5 with 1328 data
File 6: Processing chr6 with 1558 data
File 7: Processing chr7 with 1424 data
File 8: Processing chr8 with 1101 data
File 9: Processing chr9 with 1217 data
File 10: Processing chr10 with 1181 data
File 11: Processing chr11 with 1618 data
File 12: Processing chr12 with 1395 data
File 13: Processing chr13 with 542 data
File 14: Processing chr14 with 1030 data
File 15: Processing chr15 with 984 data
File 16: Processing chr16 with 1149 data
File 17: Processing chr17 with 1581 data
File 18: Processing chr18 with 454 data
File 19: Processing chr19 with 1711 data
File 20: Processing chr20 with 785 data
File 21: Processing chr21 with 419 data
File 22: Processing chr22 with 656 data
File 23: Processing chrX with 1113 data
File 24: Processing chrY with 135 data


## Combining similar chromosome

In [19]:
for chrom in chrom_list:
    print(f"Processing chromosome: {chrom}")
    file_list = []
    file_list.append(glob.glob(f"dataset/HepG2/raw/{chrom}.csv"))
    file_list.append(glob.glob(f"dataset/HepG2/raw/{chrom}_*.csv"))
    flat_file_list = flatten_concatenation(file_list)

    data = []
    for filename in flat_file_list:
        print(f"Processing file: {filename}")
        df = pd.read_csv(filename)
        data.append(df)

    df_out = pd.concat(data, axis=0, ignore_index=False)
    df_out.to_csv(f"dataset/HepG2/merged/{chrom}.csv", index=False)

print("Splitting HepG2 FINISHED!")

Processing chromosome: chr1
Processing file: dataset/HepG2/raw/chr1.csv
Processing chromosome: chr2
Processing file: dataset/HepG2/raw/chr2.csv
Processing chromosome: chr3
Processing file: dataset/HepG2/raw/chr3.csv
Processing chromosome: chr4
Processing file: dataset/HepG2/raw/chr4.csv
Processing chromosome: chr5
Processing file: dataset/HepG2/raw/chr5.csv
Processing chromosome: chr6
Processing file: dataset/HepG2/raw/chr6.csv
Processing chromosome: chr7
Processing file: dataset/HepG2/raw/chr7.csv
Processing chromosome: chr8
Processing file: dataset/HepG2/raw/chr8.csv
Processing chromosome: chr9
Processing file: dataset/HepG2/raw/chr9.csv
Processing chromosome: chr10
Processing file: dataset/HepG2/raw/chr10.csv
Processing chromosome: chr11
Processing file: dataset/HepG2/raw/chr11.csv
Processing chromosome: chr12
Processing file: dataset/HepG2/raw/chr12.csv
Processing chromosome: chr13
Processing file: dataset/HepG2/raw/chr13.csv
Processing chromosome: chr14
Processing file: dataset/He

## Changing 'chrM' into 'chrMT' (NOT REQUIRED)

In [ ]:
import os

hepg2_chrMT = pd.read_csv("dataset/HepG2/merged/chrM.csv")
hepg2_chrMT.head()

In [ ]:
hepg2_chrMT["chrom"] = "chrMT"
hepg2_chrMT.head()

In [ ]:
hepg2_chrMT.to_csv(f"dataset/HepG2/merged/chrMT.csv", index=False)

# Counting Gene Expression in the +/-2k from TSS

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

In [ ]:
def count_gene(gene_df, row, threshold = 0.8, tss_length = 4000):
    tss = row["tss"]
    hist_df = gene_df.loc[  # Inside the +/- 2k from TSS
                             (((gene_df["chromStart"] >= tss - 2000) & (gene_df["chromEnd"] <= tss + 2000)) |
                             
                             # Intersect with -2k or +2 from TSS
                             (((gene_df["chromEnd"] - (tss - 2000))/tss_length).between(threshold, 1.0)) |
                             ((((tss + 2000) - gene_df["chromStart"])/tss_length).between(threshold, 1.0))) |
                             
                             # TSS inside the gene
                             ((tss + 2000 <= gene_df["chromEnd"]) & (tss - 2000 >= gene_df["chromStart"]))
                        ]
    
    return len(hist_df.index)

In [ ]:
# Counting chr1
chrom = 'chr1'

gene_chr_df = pd.read_csv(f"dataset/HepG2/merged/{chrom}.csv")
ncbiRefSeq_chr_df = pd.read_csv(f"dataset/histone_count/{chrom}.csv")

display(gene_chr_df.head())
print(gene_chr_df.shape)
display(ncbiRefSeq_chr_df.head())
print(ncbiRefSeq_chr_df.shape)


In [ ]:
# Counting Gene
ncbiRefSeq_chr_df.loc[:, "gene_count"] = ncbiRefSeq_chr_df.progress_apply(\
                            lambda row: count_gene(gene_chr_df, row), axis=1)

In [ ]:
display(ncbiRefSeq_chr_df.head())

In [ ]:
ax = ncbiRefSeq_chr_df.hist(column=["gene_count"])

In [ ]:
ncbiRefSeq_chr_df.groupby(['gene_count']).size().reset_index(name='counts')

In [ ]:
ncbiRefSeq_chr_df.groupby(['gene_count']).size().reset_index(name='counts').\
        to_csv(f"dataset/gene_count/gene_count_{chrom}.csv", index=False)

In [ ]:
ncbiRefSeq_chr_df[(ncbiRefSeq_chr_df['gene_count'] >= 4) & (ncbiRefSeq_chr_df['histone_count'] > 20)]

In [ ]:
ncbiRefSeq_chr_df.groupby(['histone_count', 'gene_count']).size().reset_index(name='counts')

In [ ]:
ncbiRefSeq_chr_df.groupby(['histone_count', 'gene_count']).size().reset_index(name='counts').\
    to_csv(f"dataset/histone_gene_count.csv", index=False)